# 4주차 과제
빅데이터프로그래밍 · 통계학과

**이름:안석현**
**학번:5812911**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [ ]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [ ]:
# 답안

# 서로 다른 형태의 Tensor 5개 생성
# 0차원 Tensor
t1 = torch.tensor(5)

# 1차원 Tensor
t2 = torch.tensor([1.0, 2.0, 3.0])

# 2차원 Tensor
t3 = torch.tensor([[1, 2], [3, 4]])

# 3차원 Tensor
t4 = torch.ones(2, 2, 2)
# 4차원 Tensor
t5 = torch.zeros(2, 2, 2, 2, device=device)

# Tensor 이름과 데이터 정리
tensors = {
    "t1": t1,
    "t2": t2,
    "t3": t3,
    "t4": t4,
    "t5": t5
}

# 각 Tensor의 속성을 표로 만들기
data = []

for name, t in tensors.items():
    data.append({
        "이름": name,
        "shape": tuple(t.shape),
        "dim": t.dim(),
        "dtype": str(t.dtype),
        "device": str(t.device)
    })

df = pd.DataFrame(data)
print(df)


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [ ]:
# 답안

# (3, 4) 크기의 Tensor 생성
A = torch.tensor([
    [1., 2., 3., 4.],
    [5., 6., 7., 8.],
    [9., 10., 11., 12.]
])

# (4, 2) 크기의 Tensor 생성
B = torch.tensor([
    [1., 2.],
    [3., 4.],
    [5., 6.],
    [7., 8.]
])

# 1. 행렬곱 계산
C = torch.matmul(A, B)

print("행렬곱 결과:")
print(C)
print("결과 shape:", C.shape)

# 2. 첫 번째 Tensor의 열별 평균
print("열별 평균:", A.mean(dim=0))

# 3. 첫 번째 Tensor의 행별 합계
print("행별 합계:", A.sum(dim=1))


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [ ]:
# 답안

import time

# 동일한 크기의 행렬 생성
A = torch.rand(1000, 1000)
B = torch.rand(1000, 1000)

# 1. CPU에서 행렬곱 실행
torch.matmul(A, B)  # 사전 실행

start = time.time()

for i in range(5):
    C_cpu = torch.matmul(A, B)

cpu_time = (time.time() - start) / 5

print(f"CPU 평균 실행 시간: {cpu_time:.6f}초")

# 2. GPU에서 행렬곱 실행
if torch.cuda.is_available():
    A_gpu = A.to("cuda")
    B_gpu = B.to("cuda")

    # GPU 사전 실행
    torch.matmul(A_gpu, B_gpu)
    torch.cuda.synchronize()

    start = time.time()

    for i in range(5):
        C_gpu = torch.matmul(A_gpu, B_gpu)

    torch.cuda.synchronize()
    gpu_time = (time.time() - start) / 5

    print(f"GPU 평균 실행 시간: {gpu_time:.6f}초")

    # 3. CPU와 GPU 실행 시간 비교
    if gpu_time < cpu_time:
        print("GPU가 CPU보다 빠르게 연산을 처리했습니다.")
    else:
        print("이번 실험에서는 CPU가 GPU보다 빠르게 측정되었습니다.")

    print("GPU는 대규모 병렬 연산에 유리하지만, 실행 환경과 행렬 크기에 따라 차이가 발생합니다.")

else:
    print("현재 환경에서는 CUDA GPU를 사용할 수 없습니다.")
    print("GPU가 연결되지 않았거나 CUDA를 지원하지 않아 CPU만 측정했습니다.")


## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [ ]:
# 답안

# 1. y = 2x^3 + 5x, x = 3

x = torch.tensor(3.0, requires_grad=True)
y = 2 * x**3 + 5 * x

# 자동미분 실행
y.backward()

# 손으로 계산한 도함수: 6x^2 + 5
manual = 6 * (3**2) + 5

print("문제 1")
print("자동미분 결과:", x.grad.item())
print("손 계산 결과:", manual)
print("일치 여부:", x.grad.item() == manual)


# 2. 직접 만든 수식: y = x^2 + 3x, x = 2

x2 = torch.tensor(2.0, requires_grad=True)
y2 = x2**2 + 3 * x2

# 자동미분 실행
y2.backward()

# 손으로 계산한 도함수: 2x + 3
manual2 = 2 * 2 + 3

print("\n문제 2")
print("자동미분 결과:", x2.grad.item())
print("손 계산 결과:", manual2)
print("일치 여부:", x2.grad.item() == manual2)
